## Bug report #883 for virtualizarr package

### Concatenation in time of (time, y, x)-dimentional data fails

https://github.com/zarr-developers/VirtualiZarr/issues/883:


In [1]:
import obstore
import os
import virtualizarr as vz
from virtualizarr import open_virtual_dataset, open_virtual_mfdataset
from virtualizarr.parsers import HDFParser
from obspec_utils.registry import ObjectStoreRegistry
import xarray as xr

In [4]:
nc_bucket = 's3://its-live-data'
nc_dir = 'test-space/virtual-cubes'

granules = [
    "LC08_L1GT_020121_20231013_20231102_02_T2_X_LC09_L1GT_020121_20231106_20231106_02_T2_G0120V02_P084.nc",
    "LC08_L1GT_020120_20201121_20210315_02_T2_X_LC08_L1GT_020120_20210124_20210305_02_T2_G0120V02_P051.nc",
]

In [6]:
store = obstore.store.from_url(nc_bucket, region="us-west-2", skip_signature=True)
registry = ObjectStoreRegistry({nc_bucket: store})
parser = HDFParser(drop_variables=['mapping'])

all_vds = []

for granule in granules:
    nc_vds = vz.open_virtual_dataset(
        url=os.path.join(nc_bucket, nc_dir, granule),
        parser=parser,
        registry=registry,
        # loadable_variables=['time'],  # Gives more useful error message: AlignmentError: cannot reindex or align along dimension 'y' because of conflicting dimension sizes: {2560, 2048}
        loadable_variables=['time', 'y', 'x'],
        decode_times=True,
    )
    del nc_vds['img_pair_info']
    all_vds.append(nc_vds)

combined_ds = xr.concat(
    all_vds,
    dim=["time"],
    # join='outer',
    coords='minimal',
    compat='override',
    combine_attrs='override'
)


/var/folders/zt/mjfs5mx96tg275dpq3503d9m0000gq/T/ipykernel_7545/1695059955.py:19: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'x' ('x',) The recommendation is to set join explicitly for this case.
  combined_ds = xr.concat(
/var/folders/zt/mjfs5mx96tg275dpq3503d9m0000gq/T/ipykernel_7545/1695059955.py:19: FutureWarning: In a future version of xarray the default value for join will change from join='outer' to join='exact'. This change will result in the following ValueError: cannot be aligned with join='exact' because index/labels/sizes are not equal along these coordinates (dimensions): 'y' ('y',) The recommendation is to set join explicitly for this case.
  combined_ds = xr.concat(
/var/folders/zt/mjfs5mx96tg275dpq3503d9m0000gq/T/ipykernel_7545/16

NotImplementedError: Unsupported indexer. So-called 'fancy indexing' via numpy arrays is not supported, but received [[[-1]]

 [[ 0]]]